# 챗봇 화면 구현

In [ ]:
import requests
import gradio as gr
from google.colab import drive
import uuid


# 웹훅 URL
WEBHOOK_URL = "WEBHOOK_URL"

current_user_id = None

def generate_user_id():
    global current_user_id
    current_user_id = f"User_{uuid.uuid4().hex[:8]}"
    return current_user_id

# 웹훅으로 데이터를 보내는 함수
def send_to_webhook(user_message):
    try:
        data = {
            "user_id": current_user_id,
            "message": user_message
        }
        response = requests.post(WEBHOOK_URL, json=data)
        response_data = response.json()
        return response_data
    except Exception as e:
        return f"웹훅 전송 중 오류 발생: {e}"

# Gradio에 통합된 함수
def chatbot_response(user_message, history):
    global current_user_id
    if current_user_id is None:
        generate_user_id()
    response_data = send_to_webhook(user_message)  # Webhook으로부터 데이터 받기
    if isinstance(response_data, list):  # JSON 배열 변환
        bot_message = "\n\n".join([
              f"시설명: {item['시설명']}\n"
              f"도로명주소: {item['도로명주소']}\n"
              f"전화번호: {item['전화번호']}\n"
              f"운영시간: {item['운영시간']}\n"
              f"홈페이지: {item['홈페이지']}"
              for item in response_data
          ])

    else:
        bot_message = "결과를 가져오는 데 문제가 발생했습니다."
    history.append((user_message, bot_message))
    return history, ""


# 초기화 버튼을 누를 시 새로운 아이디 부여
def reset_chat():
    generate_user_id()
    return []

# Gradio 인터페이스
with gr.Blocks() as demo:
    gr.Markdown("## Peflace Bot💬")
    gr.Markdown("### 반려동물과 함께 어디든, 원하는 장소를 찾아보세요.")

    chatbot = gr.Chatbot()
    message_input = gr.Textbox(label="메시지를 입력하세요", placeholder="여기에 메시지를 입력하세요...", lines=1)
    clear_btn = gr.Button("대화 초기화")

    message_input.submit(chatbot_response, inputs=[message_input, chatbot], outputs=[chatbot, message_input])      # 메세지 입력

    clear_btn.click(reset_chat, inputs=None, outputs=chatbot, queue=False)    # 대화 초기화 버튼

demo.launch()

# 로그인 화면 구현

In [ ]:
import gradio as gr

# 가상 데이터베이스
USER_CREDENTIALS = {"test_user": "1234"}  # 아이디: 비밀번호

# 로그인 함수
def login(username, password):
    if username in USER_CREDENTIALS and USER_CREDENTIALS[username] == password:
        return gr.update(visible=False), gr.update(visible=True), ""
    else:
        return gr.update(visible=True), gr.update(visible=False), "로그인 실패: 아이디나 비밀번호가 틀렸습니다."

# 챗봇 함수
def chatbot_response(message):
    return f"Chatbot: '{message}'라고 하셨네요."

# 앱 구조
with gr.Blocks() as app:
    # 상태 메시지
    status_message = gr.Textbox(value="", label="상태 메시지", interactive=False, visible=False)

    # 로그인 화면
    with gr.Row(visible=True) as login_screen:
        username = gr.Textbox(label="아이디", placeholder="아이디 입력")
        password = gr.Textbox(label="비밀번호", placeholder="비밀번호 입력", type="password")
        login_button = gr.Button("로그인")

    # 챗봇 화면
    with gr.Row(visible=False) as chatbot_screen:
        user_input = gr.Textbox(label="메시지 입력")
        chatbot_output = gr.Textbox(label="응답", interactive=False)
        send_button = gr.Button("전송")

    # 로그인 버튼 클릭 시 화면 전환
    login_button.click(
        fn=login,
        inputs=[username, password],
        outputs=[login_screen, chatbot_screen, status_message]
    )

    # 챗봇 입력 처리
    send_button.click(
        fn=chatbot_response,
        inputs=user_input,
        outputs=chatbot_output
    )

# 앱 실행
app.launch()


# 통합 서비스 구현 

In [ ]:
import requests
import gradio as gr
import uuid

# 웹훅 URL
WEBHOOK_URL = "WEBHOOK_URL"

# 사용자 정보
user_info = {"log_id": None, "id": None, "password": None, "name": None, "email": None}

# 고유 user_id - 한 번 대화때마다 로그를 남기는게 좋을 것 같아서 일단 냅둠
def generate_log_id():
    return f"{uuid.uuid4().hex[:8]}"

# 로그인 처리
def login_process(id, password):
    if id and password:
        user_info["log_id"] = generate_log_id()
        user_info["id"] = id
        user_info["password"] = password
        return gr.update(visible=False), gr.update(visible=True)
    return "로그인 정보를 모두 입력하세요 !"

# 회원가입 처리
def register_process(id, password, name, email):
    if id and password and name and email:
        user_info["log_id"] = generate_log_id()
        user_info["id"] = id
        user_info["password"] = password
        user_info["name"] = name
        user_info["email"] = email
        return gr.update(visible=False), gr.update(visible=True)
    return "회원가입 정보를 모두 입력하세요 !"

# 웹훅으로 데이터 전송
def send_to_webhook(user_message):
    try:
        data = {
            "log_id": user_info["log_id"],
            "id": user_info["id"],
            "password": user_info["password"],
            "name": user_info.get("name"),
            "email": user_info.get("email"),
            "message": user_message
        }
        response = requests.post(WEBHOOK_URL, json=data)
        response_data = response.json()
        return response_data
    except Exception as e:
        return f"웹훅 전송 중 오류 발생: {e}"

# 챗봇 답변 변환
def chatbot_response(user_message, history):
    response_data = send_to_webhook(user_message)  # Webhook으로부터 데이터 받아옴
    if isinstance(response_data, list):  # JSON 배열 데이터 -> text로 변환
        bot_message = "\n\n".join([
            f"시설명: {item['시설명']}\n"
            f"도로명주소: {item['도로명주소']}\n"
            f"전화번호: {item['전화번호']}\n"
            f"운영시간: {item['운영시간']}\n"
            f"홈페이지: {item['홈페이지']}"
            for item in response_data
        ])
    else:
        bot_message = "결과를 가져오는 데 문제가 발생했습니다."
    history.append((user_message, bot_message))
    return history, ""

# 초기화 버튼을 누를 시 새로운 log_id 부여
def reset_chat():
    user_info["log_id"] = generate_log_id()
    return []

# Gradio 구현
with gr.Blocks() as app:
    # 로그인 화면
    with gr.Row(visible=True) as login_screen:
        with gr.Column():
            gr.Markdown("## 로그인")
            login_id = gr.Textbox(label="ID", placeholder="ID 입력")
            login_password = gr.Textbox(label="비밀번호", placeholder="비밀번호 입력", type="password")
            login_btn = gr.Button("로그인")
        with gr.Column():
            gr.Markdown("## 회원가입")
            register_id = gr.Textbox(label="ID", placeholder="ID 입력")
            register_password = gr.Textbox(label="비밀번호", placeholder="비밀번호 입력", type="password")
            register_name = gr.Textbox(label="이름", placeholder="이름 입력")
            register_email = gr.Textbox(label="이메일", placeholder="이메일 입력")
            register_btn = gr.Button("회원가입")

    # 챗봇 화면
    with gr.Row(visible=False) as chatbot_screen:
        with gr.Column():
            gr.Markdown("## Peflace Bot💬")
            gr.Markdown("### 반려동물과 함께 어디든, 원하는 장소를 찾아보세요.")
            chatbot = gr.Chatbot()
            message_input = gr.Textbox(label="메시지를 입력하세요", placeholder="여기에 메시지를 입력하세요...", lines=1)
            clear_btn = gr.Button("대화 초기화")

            message_input.submit(chatbot_response, inputs=[message_input, chatbot], outputs=[chatbot, message_input])
            clear_btn.click(reset_chat, inputs=None, outputs=chatbot)

    # 버튼 클릭 연결
    login_btn.click(
        login_process,
        inputs=[login_id, login_password],
        outputs=[login_screen, chatbot_screen]
    )
    register_btn.click(
        register_process,
        inputs=[register_id, register_password, register_name, register_email],
        outputs=[login_screen, chatbot_screen]
    )

app.launch()